# Phase 5 · zero-score submission

Writes a correctly formatted `prediction.json` (JSON list of `{"id", "pred"}`, one entry per test case) that scores **0**. No model is trained.

- `MODE = "empty"` (default): no statute for any case → zero true positives → Macro-F1 = Micro-F1 = 0 exactly.
- `MODE = "rarest"` (fallback, if Codabench rejects empty lists): every case gets only the rarest dev statute → Macro-F1 ≈ 0.0001.

**Kaggle:** CPU, secret `HF`, internet on; no extra inputs. Takes about 1 minute. Upload `P5_zero/submission.zip`.

In [ ]:
# Cell 1 — Imports and output paths for the zero-score submission.
# This notebook trains nothing; it only writes a correctly formatted prediction file.
import json, zipfile
from pathlib import Path
from collections import Counter
from huggingface_hub import snapshot_download
from kaggle_secrets import UserSecretsClient

RUN_NAME = "P5_zero"
WORK_DIR = Path("/kaggle/working") / RUN_NAME
DATA_DIR = Path("/kaggle/working/data")
WORK_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
# "empty"  → no statute for any case: zero true positives, so Macro-F1 = Micro-F1 = 0 exactly.
# "rarest" → only the rarest dev statute for every case (fallback if Codabench rejects empty lists): Macro-F1 ≈ 0.0001.
MODE = "empty"
print("Outputs:", WORK_DIR, "| mode:", MODE)

In [ ]:
# Cell 2 — Download dev.json (statute IDs and their frequencies) and test.json (case ids) using the Kaggle secret "HF".
snapshot_download(repo_id="Exploration-Lab/CS-781-Capstone", repo_type="dataset", token=UserSecretsClient().get_secret("HF"),
                  local_dir=str(DATA_DIR), allow_patterns=["Phase_5_LSI/Train/dev.json", "Phase_5_LSI/Test/test.json"])
P5_DIR = DATA_DIR / "Phase_5_LSI"
dev_frequency = Counter(int(l) for d in json.load(open(P5_DIR / "Train/dev.json")) for l in set(d["labels"]))
test_ids = [str(d["id"]) for d in json.load(open(P5_DIR / "Test/test.json"))]
print(f"{len(test_ids):,} test cases | {len(dev_frequency)} statute IDs in dev")

In [ ]:
# Cell 3 — Build the predictions: an empty statute list for every case, or (fallback) only the rarest statute.
RAREST = min(dev_frequency, key=lambda l: (dev_frequency[l], l))
if MODE == "empty":
    prediction = [{"id": i, "pred": []} for i in test_ids]
else:
    prediction = [{"id": i, "pred": [int(RAREST)]} for i in test_ids]
print(f"Every case predicted as {prediction[0]['pred']} (rarest dev statute is {RAREST}, {dev_frequency[RAREST]} dev cases)")

In [ ]:
# Cell 4 — Validate the format and build submission.zip with prediction.json at the zip root.
# One entry per test case, unique ids, integer statute IDs that exist, no duplicates within a case.
valid = set(dev_frequency)
assert len(prediction) == len(test_ids) and len({p["id"] for p in prediction}) == len(prediction)
assert all(len(set(p["pred"])) == len(p["pred"]) and set(p["pred"]) <= valid and all(isinstance(x, int) for x in p["pred"])
           for p in prediction)
json.dump(prediction, open(WORK_DIR / "prediction.json", "w"))
with zipfile.ZipFile(WORK_DIR / "submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(WORK_DIR / "prediction.json", arcname="prediction.json")
with zipfile.ZipFile(WORK_DIR / "submission.zip") as zf:
    assert zf.namelist() == ["prediction.json"]
print(f"Validated {len(prediction):,} cases -> {WORK_DIR / 'submission.zip'}")